In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"

import torch
torch.set_num_threads(4)
torch.set_num_interop_threads(1)

In [2]:
import sys

import torch

import pandas as pd

import pm4py

from config.feature_config import FeatureConfig
from config.ga_config import GAConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from ga_search.search import CounterfactualGA

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [3]:
set_seed(seed=777)

In [4]:
df = pd.read_excel(
    "../../../data/bpic20_Int.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "org:resource": "string",
        "org:role": "string",
        "case:Permit OrganizationalEntity": "string",
        "case:Amount": "float32",
        "case:RequestedAmount": "float32",
        "case:OriginalAmount": "float32",
        "case:Permit RequestedBudget": "float32",
        "case:AdjustedAmount": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [5]:
df.head(20)

,case:concept:name,time:timestamp,case:AdjustedAmount,case:Amount,case:OriginalAmount,case:Permit OrganizationalEntity,case:Permit RequestedBudget,case:RequestedAmount,concept:name,org:resource,org:role,time_delta
0,declaration 1002,2018-03-01 10:55:17,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Permit SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
1,declaration 1002,2018-03-01 10:55:21,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Permit APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,4.0
2,declaration 1002,2018-03-01 15:01:48,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Permit FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,14787.0
3,declaration 1002,2018-03-19 00:00:00,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Start trip,STAFF MEMBER,EMPLOYEE,1501092.0
4,declaration 1002,2018-03-23 00:00:00,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,End trip,STAFF MEMBER,EMPLOYEE,345600.0
5,declaration 1002,2018-03-27 16:15:02,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Declaration SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,404102.0
6,declaration 1002,2018-04-03 17:07:56,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Declaration APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,607974.0
7,declaration 1002,2018-04-05 09:45:53,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Declaration FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,146277.0
8,declaration 1002,2018-04-05 17:25:23,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Request Payment,SYSTEM,UNDEFINED,27570.0
9,declaration 1002,2018-04-09 17:30:58,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Payment Handled,SYSTEM,UNDEFINED,345935.0


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [7]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [8]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:AdjustedAmount', 'case:Amount', 'case:OriginalAmount', 'case:Permit OrganizationalEntity', 'case:Permit RequestedBudget', 'case:RequestedAmount', 'concept:name', 'org:resource', 'org:role', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [19.00, 518400.00]                       112149.0000 quantile_derived    
case:Amount                    continuous     case     yes    [28.52, 1883.08]                         375.8531   quantile_derived    
case:RequestedAmount           continuous     case    

In [9]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [10]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [11]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

### --- Process Constraints ---

In [12]:
engine = ProcessModelConstraintEngine.load(
    path = "../pretrained_models/"
)

In [13]:
engine.parallel_sets

[{'Permit APPROVED by SUPERVISOR', 'Permit FINAL_APPROVED by DIRECTOR'},
 {'Permit APPROVED by PRE_APPROVER', 'Permit FINAL_APPROVED by SUPERVISOR'}]

In [14]:
engine.branching_sets

[{'Declaration APPROVED by ADMINISTRATION',
  'Declaration APPROVED by BUDGET OWNER',
  'Declaration APPROVED by PRE_APPROVER',
  'Declaration APPROVED by SUPERVISOR',
  'Declaration FINAL_APPROVED by DIRECTOR',
  'Declaration FINAL_APPROVED by SUPERVISOR',
  'Declaration REJECTED by ADMINISTRATION',
  'Declaration REJECTED by BUDGET OWNER',
  'Declaration REJECTED by DIRECTOR',
  'Declaration REJECTED by EMPLOYEE',
  'Declaration REJECTED by MISSING',
  'Declaration REJECTED by PRE_APPROVER',
  'Declaration REJECTED by SUPERVISOR',
  'Declaration SUBMITTED by EMPLOYEE',
  'End trip',
  'Payment Handled',
  'Permit APPROVED by ADMINISTRATION',
  'Permit APPROVED by BUDGET OWNER',
  'Permit APPROVED by PRE_APPROVER',
  'Permit APPROVED by SUPERVISOR',
  'Permit FINAL_APPROVED by DIRECTOR',
  'Permit FINAL_APPROVED by SUPERVISOR',
  'Permit REJECTED by ADMINISTRATION',
  'Permit REJECTED by BUDGET OWNER',
  'Permit REJECTED by DIRECTOR',
  'Permit REJECTED by EMPLOYEE',
  'Permit REJECTE

### --- Experiments Generation ---

In [15]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic20_Int-cf_seed777_experiments_ga_ablated_output.txt", console=False)

In [16]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [17]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

### --- Counterfactuals ---

In [18]:
ga_config = GAConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=0.0,
)
ga_config.validate()

cf_GA = CounterfactualGA(
    ga_config=ga_config,
    feature_config=feature_config,
    model_wrapper=model_wrapper
)

In [19]:
results_sin = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_Ablated_single_desired_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/400 [00:00<?, ?case/s]

In [20]:
results_sin

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,declaration 69274,6,1,0,0.304601,0.139201,0.470000,0.404167,0.393333,...,0.266673,0.400000,0.100007,0.200000,0.000013,0.166667,0.0,0.000000,0.0,0.000000
1,0,declaration 916,7,1,0,0.391533,0.253065,0.530000,0.495833,0.458824,...,0.266679,0.470588,0.100012,0.200000,0.000024,0.166667,0.0,0.000000,0.0,0.000000
2,0,declaration 54601,8,1,0,0.352034,0.184067,0.520000,0.400000,0.526316,...,0.183333,0.526316,0.100000,0.200000,0.000000,0.083333,0.0,0.000000,0.0,0.000000
3,0,declaration 30359,9,1,0,0.372263,0.264526,0.480000,0.441667,0.442857,...,0.183333,0.571429,0.100000,0.200000,0.000000,0.083333,0.0,0.000000,0.0,0.000000
4,0,declaration 34740,10,1,0,0.355568,0.351136,0.360000,0.433333,0.478261,...,0.154762,0.347826,0.071429,0.000000,0.142857,0.083333,0.0,0.000000,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
347,38,declaration 56837,21,1,0,0.432017,0.416414,0.447619,0.543056,0.444444,...,0.237698,0.444444,0.098810,0.095238,0.102381,0.138889,0.0,0.000000,0.0,0.000000
348,38,declaration 51709,21,1,0,0.278191,0.206383,0.350000,0.423611,0.444444,...,0.157229,0.444444,0.046118,0.047619,0.044618,0.111111,0.0,0.000000,0.0,0.000000
349,38,declaration 20507,22,1,0,0.363621,0.324861,0.402381,0.584722,0.468085,...,0.457059,0.468085,0.123726,0.095238,0.152214,0.333333,0.0,0.902804,0.0,1.000000
350,38,declaration 4604,23,1,0,0.444869,0.477832,0.411905,0.552778,0.304082,...,0.167714,0.244898,0.056603,0.047619,0.065587,0.111111,0.0,0.000000,0.0,0.000000


### --- Cleanup ---

In [21]:
sys.stdout = original_stdout
log_file.close()